# NAMAA V5 Kaggle
Use `virtualenv` rather than Debian `venv`/`ensurepip`, then match torch/torchaudio 2.6.0 CUDA12.6 in isolated process. Reports WAV if generated.

In [ ]:
import os,sys,subprocess
from pathlib import Path
env=os.environ.copy()
env["PIP_NO_CACHE_DIR"]="1"
env["HF_HOME"]="/kaggle/working/namaa_hf_cache"
venv="/kaggle/working/namaa_venv_v5"
print("V5_BOOTSTRAP_VIRTUALENV",flush=True)
subprocess.run([sys.executable,"-m","pip","install","-q","virtualenv"],check=True,env=env)
print("V5_CREATE_ISOLATED",flush=True)
subprocess.run([sys.executable,"-m","virtualenv","--clear","--no-download",venv],check=True,env=env)
py=venv+"/bin/python"
print("V5_INSTALL_MATCHED_TORCH_CU126",flush=True)
subprocess.run([py,"-m","pip","install","-q","torch==2.6.0","torchaudio==2.6.0","--index-url","https://download.pytorch.org/whl/cu126"],check=True,env=env)
print("V5_INSTALL_CHATTERBOX",flush=True)
subprocess.run([py,"-m","pip","install","-q","chatterbox-tts==0.1.7","huggingface_hub"],check=True,env=env)
print("V5_VERIFY_NEW_PROCESS",flush=True)
subprocess.run([py,"-c","import torch,torchaudio;from chatterbox.mtl_tts import ChatterboxMultilingualTTS;print('PRECHECK_OK',torch.__version__,torchaudio.__version__,torch.cuda.is_available(),flush=True);assert torch.cuda.is_available()"],check=True,env=env)
s=Path("/kaggle/working/namaa_infer_v5.py")
s.write_text("import time\nfrom pathlib import Path\nimport torch\nimport torchaudio\nfrom huggingface_hub import snapshot_download\nfrom safetensors.torch import load_file\nfrom chatterbox.mtl_tts import ChatterboxMultilingualTTS\nprint(\"PREFLIGHT\",torch.__version__,torchaudio.__version__,torch.cuda.is_available(),flush=True)\nassert torch.__version__.split(\"+\")[0]==\"2.6.0\"\nassert torchaudio.__version__.split(\"+\")[0]==\"2.6.0\"\nassert torch.cuda.is_available()\nt=time.monotonic()\nprint(\"LOAD_BASE\",flush=True)\nmodel=ChatterboxMultilingualTTS.from_pretrained(device=\"cuda\")\nprint(\"LOAD_NAMAA\",flush=True)\nckpt=snapshot_download(\"NAMAA-Space/NAMAA-Egyptian-TTS\",allow_patterns=[\"t3_mtl23ls_v2.safetensors\"])\nmodel.t3.load_state_dict(load_file(str(Path(ckpt)/\"t3_mtl23ls_v2.safetensors\"),device=\"cpu\"),strict=True)\nmodel.t3.to(\"cuda\").eval()\nprint(\"NAMAA_READY\",round(time.monotonic()-t,1),flush=True)\nbegin=time.monotonic()\nwav=model.generate(\"يا أهلا بيكم. النهارده هنتكلم عن مانسا موسى، واحد من أغنى ملوك التاريخ.\",language_id=\"ar\")\nout=Path(\"/kaggle/working/namaa_test.wav\")\ntorchaudio.save(str(out),wav.detach().cpu(),model.sr)\nprint(\"SUCCESS\",str(out),\"BYTES\",out.stat().st_size,\"AUDIO_SECONDS\",round(wav.shape[-1]/model.sr,2),\"INFER_SECONDS\",round(time.monotonic()-begin,2),\"TOTAL_SECONDS\",round(time.monotonic()-t,2),flush=True)\n",encoding="utf-8")
print("V5_START_INFERENCE",flush=True)
subprocess.run([py,"-u",str(s)],check=True,env=env)
